# Olist Customer Intelligence System — Data Cleaning

This notebook cleans all nine Olist datasets and prepares them for Stage 1 and Stage 2 of the project.

**Principle: investigate first, clean second.**  
We only remove or change data when we have a clear reason. If something is uncertain, we keep it and explain the issue.

**Inputs:** nine raw CSV files in the `Raw_Data` folder  
**Outputs:** nine cleaned CSV files in the `Cleaned_Data` folder, plus a cleaning summary  

Run the cells from top to bottom.

## 1. Imports and output folder

In [55]:
import pandas as pd
import numpy as np
import os

# Create the output folder if it doesn't already exist.
os.makedirs("Cleaned_Data", exist_ok=True)

print("Imports done. Cleaned_Data folder is ready.")

Imports done. Cleaned_Data folder is ready.


## 2. Load the nine raw datasets

We load every file into its own variable with a simple, clear name.  
We will **not** change these raw DataFrames — all cleaning happens on copies.

In [56]:
customers_raw          = pd.read_csv("Raw_Data/olist_customers_dataset.csv")
geolocation_raw        = pd.read_csv("Raw_Data/olist_geolocation_dataset.csv")
order_items_raw        = pd.read_csv("Raw_Data/olist_order_items_dataset.csv")
payments_raw           = pd.read_csv("Raw_Data/olist_order_payments_dataset.csv")
reviews_raw            = pd.read_csv("Raw_Data/olist_order_reviews_dataset.csv")
orders_raw             = pd.read_csv("Raw_Data/olist_orders_dataset.csv")
products_raw           = pd.read_csv("Raw_Data/olist_products_dataset.csv")
sellers_raw            = pd.read_csv("Raw_Data/olist_sellers_dataset.csv")
category_translation_raw = pd.read_csv("Raw_Data/product_category_name_translation.csv")

print("All nine CSV files loaded successfully.")

All nine CSV files loaded successfully.


### Quick row-count snapshot

This gives us a baseline to compare against after cleaning.

In [57]:
raw_shapes = {
    "customers":           customers_raw.shape,
    "geolocation":         geolocation_raw.shape,
    "order_items":         order_items_raw.shape,
    "payments":            payments_raw.shape,
    "reviews":             reviews_raw.shape,
    "orders":              orders_raw.shape,
    "products":            products_raw.shape,
    "sellers":             sellers_raw.shape,
    "category_translation": category_translation_raw.shape,
}

for name, shape in raw_shapes.items():
    print(f"{name:25s}  {shape[0]:>8,} rows  x  {shape[1]} columns")

customers                    99,441 rows  x  5 columns
geolocation                1,000,163 rows  x  5 columns
order_items                 112,650 rows  x  7 columns
payments                    103,886 rows  x  5 columns
reviews                      99,224 rows  x  7 columns
orders                       99,441 rows  x  8 columns
products                     32,951 rows  x  9 columns
sellers                       3,095 rows  x  4 columns
category_translation             71 rows  x  2 columns


## 3. Make working copies

We work on copies so the originals stay untouched.  
The `_raw` variables always hold the original data.

In [58]:
customers            = customers_raw.copy()
geolocation          = geolocation_raw.copy()
order_items          = order_items_raw.copy()
payments             = payments_raw.copy()
reviews              = reviews_raw.copy()
orders               = orders_raw.copy()
products             = products_raw.copy()
sellers              = sellers_raw.copy()
category_translation = category_translation_raw.copy()

print("Working copies created.")

Working copies created.


---
## 4. Customers

**What we know from exploration:**
- 99,441 rows, 0 duplicates, 0 missing values.
- `customer_id` is unique per row — it represents one purchase session, not one person.
- `customer_unique_id` is the actual person. 2,997 customers have placed more than one order.

We do not delete rows just because a `customer_unique_id` appears more than once. That is expected and useful data.

### 4a. Check for duplicate rows

In [59]:
print("Exact duplicate rows:", customers.duplicated().sum())
print("Duplicate customer_id:", customers['customer_id'].duplicated().sum())

Exact duplicate rows: 0
Duplicate customer_id: 0


### 4b. Check for missing values

In [60]:
print(customers.isna().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


### 4c. Understand returning customers

`customer_unique_id` identifies the real person. A value appearing more than once means that person placed multiple orders.

In [61]:
unique_people = customers['customer_unique_id'].nunique()
total_rows    = len(customers)
returning     = (customers['customer_unique_id'].value_counts() > 1).sum()

print(f"Total rows (order-customer links): {total_rows:,}")
print(f"Distinct real customers:           {unique_people:,}")
print(f"Customers who ordered more than once: {returning:,}")

Total rows (order-customer links): 99,441
Distinct real customers:           96,096
Customers who ordered more than once: 2,997


### 4d. Cleaning decision for customers

The customers table is already clean — no duplicates, no missing values, correct data types.  
**Action:** no rows removed. We trim any whitespace from text columns as a general good practice.

In [62]:
# Strip leading/trailing spaces from all text columns.
customers['customer_city']  = customers['customer_city'].str.strip()
customers['customer_state'] = customers['customer_state'].str.strip()

print("Customers after cleaning:", customers.shape)
print("Rows removed:", len(customers_raw) - len(customers))

Customers after cleaning: (99441, 5)
Rows removed: 0


---
## 5. Geolocation

**What we know from exploration:**
- 1,000,163 rows.
- 261,831 exact duplicate rows (same ZIP code, same lat, same lng, same city, same state).
- A small number of coordinates fall outside Brazil's geographic boundaries.

Brazil's approximate bounding box:
- Latitude:  -35° (south) to +5° (north)
- Longitude: -75° (west) to -28° (east)

Multiple rows per ZIP code with **different** coordinates are normal — a ZIP covers an area.

### 5a. Check exact duplicates

In [63]:
print("Rows before removing duplicates:", len(geolocation))
print("Exact duplicate rows:           ", geolocation.duplicated().sum())

Rows before removing duplicates: 1000163
Exact duplicate rows:            261831


### 5b. Remove exact duplicate rows

A row is a true duplicate when every column — ZIP, latitude, longitude, city, and state — is identical.  
Keeping only the first occurrence of each completely identical row is safe.

In [64]:
geolocation = geolocation.drop_duplicates()

print("Rows after removing exact duplicates:", len(geolocation))
print("Rows removed:", len(geolocation_raw) - len(geolocation))

Rows after removing exact duplicates: 738332
Rows removed: 261831


### 5c. Check for coordinates outside Brazil

Any row with latitude above 5° or below -35°, or longitude outside -75° to -28°, is almost certainly an error.

In [65]:
# Flag rows where coordinates are outside Brazil's bounding box.
bad_lat = (geolocation['geolocation_lat'] > 5) | (geolocation['geolocation_lat'] < -35)
bad_lng = (geolocation['geolocation_lng'] > -28) | (geolocation['geolocation_lng'] < -75)
bad_coords = bad_lat | bad_lng

print("Rows with coordinates outside Brazil:", bad_coords.sum())
print()
print(geolocation[bad_coords][['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
                                'geolocation_city', 'geolocation_state']].head(10))

Rows with coordinates outside Brazil: 25

        geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
387565                        18243        28.008978       -15.536867   
513631                        28165        41.614052        -8.411675   
513754                        28155        42.439286        13.820214   
514429                        28333        38.381672        -6.328200   
516682                        28595        43.684961        -7.411080   
538512                        29654        29.409252       -98.484121   
538557                        29654        21.657547      -101.466766   
585242                        35179        25.995203       -98.078544   
585260                        35179        25.995245       -98.078533   
695377                        45936        38.323939        -6.775035   

               geolocation_city geolocation_state  
387565  bom retiro da esperanca                SP  
513631      vila nova de campos                RJ 

### 5d. Remove out-of-bounds coordinates

These coordinates are clearly wrong — latitude values like 45° or longitude like 121° cannot be in Brazil.  
We remove them because they would corrupt any distance or map analysis.

In [66]:
rows_before = len(geolocation)
geolocation = geolocation[~bad_coords].copy()

print("Rows before:", rows_before)
print("Rows after: ", len(geolocation))
print("Rows removed (bad coordinates):", rows_before - len(geolocation))

Rows before: 738332
Rows after:  738307
Rows removed (bad coordinates): 25


### 5e. Strip whitespace from city names

In [67]:
geolocation['geolocation_city']  = geolocation['geolocation_city'].str.strip()
geolocation['geolocation_state'] = geolocation['geolocation_state'].str.strip()

print("Geolocation final shape:", geolocation.shape)

Geolocation final shape: (738307, 5)


---
## 6. Order items

**What we know from exploration:**
- 112,650 rows, 0 exact duplicates, 0 missing values.
- One order can contain multiple products. The composite key is `(order_id, order_item_id)`.
- Minimum price is 0.85 — plausible. Minimum freight is 0.00 — also plausible (free shipping promotions).
- All `order_id`, `product_id`, and `seller_id` values match their parent tables.
- `shipping_limit_date` is stored as a plain string (object) and needs to be converted to datetime.

### 6a. Check duplicates and missing values

In [68]:
print("Exact duplicate rows:", order_items.duplicated().sum())
print()
print("Missing values per column:")
print(order_items.isna().sum())

Exact duplicate rows: 0

Missing values per column:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


### 6b. Check that the composite key is unique

Each row should be uniquely identified by `(order_id, order_item_id)`.  
For example, order X item 1, order X item 2, etc.

In [69]:
composite_dupes = order_items.duplicated(subset=['order_id', 'order_item_id']).sum()
print("Duplicate (order_id, order_item_id) combinations:", composite_dupes)

Duplicate (order_id, order_item_id) combinations: 0


### 6c. Check price and freight ranges

In [70]:
print("Price range:")
print(order_items['price'].describe())
print()
print("Freight range:")
print(order_items['freight_value'].describe())

Price range:
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Freight range:
count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64


### 6d. Convert shipping_limit_date to datetime

In [71]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

print("shipping_limit_date dtype:", order_items['shipping_limit_date'].dtype)
print("Order items final shape:  ", order_items.shape)
print("Rows removed:              0 (table was already clean)")

shipping_limit_date dtype: datetime64[us]
Order items final shape:   (112650, 7)
Rows removed:              0 (table was already clean)


---
## 7. Payments

**What we know from exploration:**
- 103,886 rows, 0 exact duplicates, 0 missing values.
- One order can have **multiple** payment rows (e.g. part credit card + part voucher). This is normal.
- 9 rows have `payment_value = 0.00`.
- 2 rows have `payment_installments = 0` (both are credit card payments).
- 3 rows have `payment_type = 'not_defined'`.

We will investigate each anomaly before deciding what to do.

### 7a. Check payment_type values

In [72]:
print(payments['payment_type'].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


### 7b. Look at the three 'not_defined' rows

In [73]:
# What do these rows look like?
not_defined = payments[payments['payment_type'] == 'not_defined']
print(not_defined)

                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  


### 7c. Look at the zero-value payments

In [74]:
zero_value = payments[payments['payment_value'] == 0]
print(f"Rows with payment_value = 0: {len(zero_value)}")
print()
print(zero_value[['order_id', 'payment_type', 'payment_installments', 'payment_value']])

Rows with payment_value = 0: 9

                                order_id payment_type  payment_installments  \
19922   8bcbe01d44d147f901cd3192671144db      voucher                     1   
36822   fa65dad1b0e818e3ccc5cb0e39231352      voucher                     1   
43744   6ccb433e00daae1283ccc956189c82ae      voucher                     1   
51280   4637ca194b6387e2d538dc89b124b0ee  not_defined                     1   
57411   00b1cb0320190ca0daa2c88b35206009  not_defined                     1   
62674   45ed6e85398a87c253db47c2d9f48216      voucher                     1   
77885   fa65dad1b0e818e3ccc5cb0e39231352      voucher                     1   
94427   c8c528189310eaa44a745b8d9d26908b  not_defined                     1   
100766  b23878b3e8eb4d25a158f57d96331b18      voucher                     1   

        payment_value  
19922             0.0  
36822             0.0  
43744             0.0  
51280             0.0  
57411             0.0  
62674             0.0  
77885    

### 7d. Look at the zero-installment rows

In [75]:
zero_install = payments[payments['payment_installments'] == 0]
print(f"Rows with payment_installments = 0: {len(zero_install)}")
print()
print(zero_install)

Rows with payment_installments = 0: 2

                               order_id  payment_sequential payment_type  \
46982  744bade1fcf9ff3f31d860ace076d422                   2  credit_card   
79014  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card   

       payment_installments  payment_value  
46982                     0          58.69  
79014                     0         129.94  


### 7e. Cleaning decision for payments

- The `not_defined` rows have a valid `payment_value`. We cannot safely infer the correct type. **Keep and flag.**
- The zero-value rows likely represent vouchers or free-shipping adjustments. We cannot verify the correct value. **Keep and flag.**
- The two credit-card rows with 0 installments are suspicious but we cannot determine the correct value. **Keep and flag.**

**Action:** no rows deleted. We strip whitespace from the `payment_type` column.

In [76]:
payments['payment_type'] = payments['payment_type'].str.strip()

print("Payments final shape:", payments.shape)
print("Rows removed: 0")
print()
print("UNRESOLVED: 9 rows with payment_value=0, 2 with installments=0, 3 with type='not_defined'. Keep and flag.")

Payments final shape: (103886, 5)
Rows removed: 0

UNRESOLVED: 9 rows with payment_value=0, 2 with installments=0, 3 with type='not_defined'. Keep and flag.


---
## 8. Reviews

**What we know from exploration:**
- 99,224 rows, 0 exact duplicate rows.
- 87,656 missing `review_comment_title` — most customers do not leave a title. This is normal.
- 58,247 missing `review_comment_message` — also normal.
- 789 `review_id` values appear on **different** `order_id`s. These are not exact duplicates — each row has a unique `order_id`. This looks like a system issue where the same review ID was reused across different orders. We **do not delete** these rows because each one belongs to a real order.

### 8a. Check exact duplicates

In [77]:
print("Exact duplicate rows:", reviews.duplicated().sum())

Exact duplicate rows: 0


### 8b. Investigate repeated review_id values

A `review_id` appears more than once. Let's confirm these are different orders (not true duplicates).

In [78]:
repeated_ids = reviews['review_id'].value_counts()
print("review_id values that appear more than once:", (repeated_ids > 1).sum())
print()

# Show a few examples. Each repeated review_id has a DIFFERENT order_id — so these are real records.
sample_id = repeated_ids[repeated_ids > 1].index[0]
print(f"Example — review_id '{sample_id}':")
print(reviews[reviews['review_id'] == sample_id][['review_id', 'order_id', 'review_score']])

review_id values that appear more than once: 789

Example — review_id 'c444278834184f72b1484dfe47de7f97':
                              review_id                          order_id  \
2952   c444278834184f72b1484dfe47de7f97  df56136b8031ecd28e200bb18e6ddb2e   
43897  c444278834184f72b1484dfe47de7f97  566f53fc7d36fa366f7e221468121877   
95774  c444278834184f72b1484dfe47de7f97  8557dabbdacec1a9e250f5e70afe1eab   

       review_score  
2952              5  
43897             5  
95774             5  


### 8c. Check review scores and missing fields

In [79]:
print("Review score distribution:")
print(reviews['review_score'].value_counts().sort_index())
print()
print("Missing values per column:")
print(reviews.isna().sum())

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Missing values per column:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


### 8d. Convert date columns to datetime

In [80]:
reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print("Date columns converted:")
print(reviews[['review_creation_date', 'review_answer_timestamp']].dtypes)

Date columns converted:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


### 8e. Cleaning decision for reviews

- 0 exact duplicate rows — nothing to remove.
- Repeated `review_id` values are attached to different orders; we keep all rows.
- Missing comment title and message are expected (optional fields).
- Scores 1–5 all present — no invalid values.

**Action:** no rows deleted. Date columns converted. Strip whitespace from any text fields.

In [81]:
reviews['review_comment_title']   = reviews['review_comment_title'].str.strip()
reviews['review_comment_message'] = reviews['review_comment_message'].str.strip()

print("Reviews final shape:", reviews.shape)
print("Rows removed: 0")
print()
print("NOTE: 789 review_id values appear on different orders. These are kept as-is.")

Reviews final shape: (99224, 7)
Rows removed: 0

NOTE: 789 review_id values appear on different orders. These are kept as-is.


---
## 9. Orders

**What we know from exploration:**
- 99,441 rows, 0 exact duplicates.
- Missing values are concentrated in timestamp columns:
  - `order_approved_at`: 160 missing — almost all are cancelled or 'created' orders.
  - `order_delivered_carrier_date`: 1,783 missing — mostly non-delivered statuses.
  - `order_delivered_customer_date`: 2,965 missing — mostly shipped, cancelled, or other non-delivered statuses.
- All timestamp columns are stored as strings and need to be converted to datetime.
- **We do not fill missing dates with invented values.** Missing delivery dates for shipped/cancelled orders are expected.

### 9a. Check missing values by column

In [82]:
print("Missing values per column:")
print(orders.isna().sum())

Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


### 9b. Understand why order_approved_at is missing

We check which order statuses have a missing `order_approved_at`. If the missing values match non-delivered statuses, they make sense.

In [83]:
print("Missing order_approved_at by order_status:")
print(orders[orders['order_approved_at'].isna()]['order_status'].value_counts())

Missing order_approved_at by order_status:
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


### 9c. Understand why order_delivered_customer_date is missing

In [84]:
print("Missing order_delivered_customer_date by order_status:")
print(orders[orders['order_delivered_customer_date'].isna()]['order_status'].value_counts())

Missing order_delivered_customer_date by order_status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


### 9d. Understand why order_delivered_carrier_date is missing

In [85]:
print("Missing order_delivered_carrier_date by order_status:")
print(orders[orders['order_delivered_carrier_date'].isna()]['order_status'].value_counts())

Missing order_delivered_carrier_date by order_status:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


**Interpretation:** Almost all missing timestamps belong to orders that were never completed — they were cancelled, unavailable, invoiced, or still in processing. This is expected behaviour, not a data error. We keep these rows.

### 9e. Convert all timestamp columns to datetime

In [86]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date column dtypes after conversion:")
print(orders[date_cols].dtypes)

Date column dtypes after conversion:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


### 9f. Check for suspicious timestamp sequences

A delivered order should have: purchase → approved → carrier → delivered.  
We flag cases where the customer delivery date is before the purchase date.

In [87]:
# Only check rows where both timestamps are available.
both_present = orders.dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date'])
reversed_dates = both_present[
    both_present['order_delivered_customer_date'] < both_present['order_purchase_timestamp']
]
print(f"Orders where delivery date is before purchase date: {len(reversed_dates)}")

Orders where delivery date is before purchase date: 0


### 9g. Cleaning decision for orders

- 0 exact duplicates — nothing to remove.
- Missing timestamps are expected for non-delivered orders — keep them as NaT.
- No reversed delivery dates found.

**Action:** no rows deleted. All timestamp columns converted to datetime.

In [88]:
orders['order_status'] = orders['order_status'].str.strip()

print("Orders final shape:", orders.shape)
print("Rows removed: 0")

Orders final shape: (99441, 8)
Rows removed: 0


---
## 10. Products

**What we know from exploration:**
- 32,951 rows, 0 exact duplicates.
- 610 rows are missing `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` (all the same 610 rows).
- 2 rows are missing `product_weight_g` and all dimension columns.
- 4 rows have `product_weight_g = 0`, which is physically impossible for a shipped item.
- Two product categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) are not in the translation table.

Column name note: the original file has `product_name_lenght` (typo for 'length'). We keep it as-is to match the original.

### 10a. Check missing values

In [89]:
print("Missing values per column:")
print(products.isna().sum())

Missing values per column:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


### 10b. Investigate the 610 rows missing category and text attributes

Are these products still referenced by order items? If yes, we keep them — they are real products with missing descriptive info.

In [90]:
missing_category = products[products['product_category_name'].isna()]
print(f"Products missing category: {len(missing_category)}")

# Check if these products appear in order_items.
missing_ids_in_orders = missing_category['product_id'].isin(order_items['product_id']).sum()
print(f"Of these, products that appear in order_items: {missing_ids_in_orders}")
print()
print("Sample rows with missing category:")
print(missing_category[['product_id', 'product_category_name', 'product_weight_g']].head())

Products missing category: 610
Of these, products that appear in order_items: 610

Sample rows with missing category:
                           product_id product_category_name  product_weight_g
105  a41e356c76fab66334f36de622ecbd3a                   NaN             650.0
128  d8dee61c2034d6d075997acef1870e9b                   NaN             300.0
145  56139431d72cd51f19eb9f7dae4d1617                   NaN             200.0
154  46b48281eb6d663ced748f324108c733                   NaN           18500.0
197  5fb61f482620cb672f5e586bb132eae9                   NaN             300.0


### 10c. Investigate zero-weight products

In [91]:
zero_weight = products[products['product_weight_g'] == 0]
print(f"Products with weight = 0g: {len(zero_weight)}")
print()
print(zero_weight[['product_id', 'product_category_name', 'product_weight_g',
                     'product_length_cm', 'product_height_cm', 'product_width_cm']])

Products with weight = 0g: 4

                             product_id product_category_name  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho   
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho   
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho   
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho   

       product_weight_g  product_length_cm  product_height_cm  \
9769                0.0               30.0               25.0   
13683               0.0               30.0               25.0   
14997               0.0               30.0               25.0   
32079               0.0               30.0               25.0   

       product_width_cm  
9769               30.0  
13683              30.0  
14997              30.0  
32079              30.0  


### 10d. Check categories not in the translation table

In [92]:
cats_in_products = set(products['product_category_name'].dropna().unique())
cats_in_trans    = set(category_translation['product_category_name'].unique())

not_translated = cats_in_products - cats_in_trans
print(f"Categories in products but not in translation table ({len(not_translated)}):")
for c in sorted(not_translated):
    count = (products['product_category_name'] == c).sum()
    print(f"  {c}  ({count} products)")

Categories in products but not in translation table (2):
  pc_gamer  (3 products)
  portateis_cozinha_e_preparadores_de_alimentos  (10 products)


### 10e. Cleaning decision for products

- **610 missing category rows**: These are real products sold in real orders. We keep them. The missing category will show as `NaN` in any category analysis.
- **4 zero-weight products**: A weight of 0g is physically impossible for a shipped item. We cannot invent the correct weight, so we set them to `NaN` and flag them.
- **2 missing weight/dimension rows**: Already `NaN`. We leave them as-is.
- **2 untranslated categories**: We keep the original Portuguese names. Future analysis should handle them gracefully.

**Action:** set 4 zero weights to NaN. No rows deleted.

In [93]:
# Replace weight = 0 with NaN because 0g is not a valid shipping weight.
zero_weight_mask = products['product_weight_g'] == 0
print(f"Setting {zero_weight_mask.sum()} zero-weight values to NaN.")
products.loc[zero_weight_mask, 'product_weight_g'] = np.nan

print()
print("product_weight_g missing after fix:", products['product_weight_g'].isna().sum())
print("Products final shape:", products.shape)
print("Rows removed: 0")

Setting 4 zero-weight values to NaN.

product_weight_g missing after fix: 6
Products final shape: (32951, 9)
Rows removed: 0


---
## 11. Sellers

**What we know from exploration:**
- 3,095 rows. Seller_id is unique (3,095 unique values). 0 duplicates, 0 missing values.
- The table is clean. We only strip whitespace from text columns.

In [94]:
print("Exact duplicate rows:", sellers.duplicated().sum())
print("Missing values:")
print(sellers.isna().sum())

Exact duplicate rows: 0
Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


In [95]:
sellers['seller_city']  = sellers['seller_city'].str.strip()
sellers['seller_state'] = sellers['seller_state'].str.strip()

print("Sellers final shape:", sellers.shape)
print("Rows removed: 0")

Sellers final shape: (3095, 4)
Rows removed: 0


---
## 12. Category translation

**What we know from exploration:**
- 71 rows, 0 duplicates, 0 missing values.
- 2 categories in the products table are not covered by this table (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`).
- We **preserve the original Portuguese category names**. We do not rename or delete anything.

In [96]:
print("Exact duplicate rows:", category_translation.duplicated().sum())
print("Missing values:")
print(category_translation.isna().sum())
print()
print("Category translation final shape:", category_translation.shape)
print("Rows removed: 0")

Exact duplicate rows: 0
Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Category translation final shape: (71, 2)
Rows removed: 0


---
## 13. Validate relationships between datasets

We check that foreign keys actually match their parent table. This tells us whether the data is internally consistent.

### 13a. Orders → Customers

In [97]:
valid_customers = set(customers['customer_id'])
bad_cust = orders[~orders['customer_id'].isin(valid_customers)]
print(f"Orders referencing a customer_id not in customers table: {len(bad_cust)}")

Orders referencing a customer_id not in customers table: 0


### 13b. Order items → Orders, Products, Sellers

In [98]:
valid_orders   = set(orders['order_id'])
valid_products = set(products['product_id'])
valid_sellers  = set(sellers['seller_id'])

bad_items_orders   = order_items[~order_items['order_id'].isin(valid_orders)]
bad_items_products = order_items[~order_items['product_id'].isin(valid_products)]
bad_items_sellers  = order_items[~order_items['seller_id'].isin(valid_sellers)]

print(f"Order items with unknown order_id:   {len(bad_items_orders)}")
print(f"Order items with unknown product_id: {len(bad_items_products)}")
print(f"Order items with unknown seller_id:  {len(bad_items_sellers)}")

Order items with unknown order_id:   0
Order items with unknown product_id: 0
Order items with unknown seller_id:  0


### 13c. Payments → Orders

In [99]:
bad_payments = payments[~payments['order_id'].isin(valid_orders)]
print(f"Payment rows referencing unknown order_id: {len(bad_payments)}")

Payment rows referencing unknown order_id: 0


### 13d. Reviews → Orders

In [100]:
bad_reviews = reviews[~reviews['order_id'].isin(valid_orders)]
print(f"Review rows referencing unknown order_id: {len(bad_reviews)}")

Review rows referencing unknown order_id: 0


### 13e. Orders with no payment record

In [101]:
orders_with_payment = set(payments['order_id'])
orders_no_payment = orders[~orders['order_id'].isin(orders_with_payment)]
print(f"Orders with no payment record: {len(orders_no_payment)}")
if len(orders_no_payment) > 0:
    print()
    print(orders_no_payment[['order_id', 'order_status']])

Orders with no payment record: 1

                               order_id order_status
30710  bfbd0f9bdef84302105ad712db648a6c    delivered


**Relationship summary:**  
All foreign keys are valid. The one delivered order with no payment record is an anomaly we cannot resolve from this data. We keep it.

---
## 14. Save cleaned datasets

We save each cleaned DataFrame to the `Cleaned_Data` folder using the same filename as the original.  
We also print the before/after row counts so you can see exactly what changed.

In [102]:
customers.to_csv("Cleaned_Data/olist_customers_dataset.csv", index=False)
geolocation.to_csv("Cleaned_Data/olist_geolocation_dataset.csv", index=False)
order_items.to_csv("Cleaned_Data/olist_order_items_dataset.csv", index=False)
payments.to_csv("Cleaned_Data/olist_order_payments_dataset.csv", index=False)
reviews.to_csv("Cleaned_Data/olist_order_reviews_dataset.csv", index=False)
orders.to_csv("Cleaned_Data/olist_orders_dataset.csv", index=False)
products.to_csv("Cleaned_Data/olist_products_dataset.csv", index=False)
sellers.to_csv("Cleaned_Data/olist_sellers_dataset.csv", index=False)
category_translation.to_csv("Cleaned_Data/product_category_name_translation.csv", index=False)

print("All nine cleaned files saved to Cleaned_Data/.")

All nine cleaned files saved to Cleaned_Data/.


### Save a cleaning summary CSV

This small file records what changed — useful for documentation and for anyone reviewing the project.

In [103]:
# Collect the raw and cleaned shapes for each dataset.
summary_rows = [
    {"dataset": "customers",
     "rows_raw": len(customers_raw),
     "rows_clean": len(customers),
     "rows_removed": len(customers_raw) - len(customers),
     "missing_remaining": customers.isna().sum().sum()},

    {"dataset": "geolocation",
     "rows_raw": len(geolocation_raw),
     "rows_clean": len(geolocation),
     "rows_removed": len(geolocation_raw) - len(geolocation),
     "missing_remaining": geolocation.isna().sum().sum()},

    {"dataset": "order_items",
     "rows_raw": len(order_items_raw),
     "rows_clean": len(order_items),
     "rows_removed": len(order_items_raw) - len(order_items),
     "missing_remaining": order_items.isna().sum().sum()},

    {"dataset": "payments",
     "rows_raw": len(payments_raw),
     "rows_clean": len(payments),
     "rows_removed": len(payments_raw) - len(payments),
     "missing_remaining": payments.isna().sum().sum()},

    {"dataset": "reviews",
     "rows_raw": len(reviews_raw),
     "rows_clean": len(reviews),
     "rows_removed": len(reviews_raw) - len(reviews),
     "missing_remaining": reviews.isna().sum().sum()},

    {"dataset": "orders",
     "rows_raw": len(orders_raw),
     "rows_clean": len(orders),
     "rows_removed": len(orders_raw) - len(orders),
     "missing_remaining": orders.isna().sum().sum()},

    {"dataset": "products",
     "rows_raw": len(products_raw),
     "rows_clean": len(products),
     "rows_removed": len(products_raw) - len(products),
     "missing_remaining": products.isna().sum().sum()},

    {"dataset": "sellers",
     "rows_raw": len(sellers_raw),
     "rows_clean": len(sellers),
     "rows_removed": len(sellers_raw) - len(sellers),
     "missing_remaining": sellers.isna().sum().sum()},

    {"dataset": "category_translation",
     "rows_raw": len(category_translation_raw),
     "rows_clean": len(category_translation),
     "rows_removed": len(category_translation_raw) - len(category_translation),
     "missing_remaining": category_translation.isna().sum().sum()},
]

summary = pd.DataFrame(summary_rows)
summary.to_csv("Cleaned_Data/cleaning_summary.csv", index=False)

print("cleaning_summary.csv saved.")
print()
summary

cleaning_summary.csv saved.



,dataset,rows_raw,rows_clean,rows_removed,missing_remaining
0,customers,99441,99441,0,0
1,geolocation,1000163,738307,261856,0
2,order_items,112650,112650,0,0
3,payments,103886,103886,0,0
4,reviews,99224,99224,0,145903
5,orders,99441,99441,0,4908
6,products,32951,32951,0,2452
7,sellers,3095,3095,0,0
8,category_translation,71,71,0,0


---
## 15. Final validation

We run a final check on every cleaned dataset to confirm the results.

### 15a. Shape of each cleaned dataset

In [104]:
datasets = {
    "customers":           customers,
    "geolocation":         geolocation,
    "order_items":         order_items,
    "payments":            payments,
    "reviews":             reviews,
    "orders":              orders,
    "products":            products,
    "sellers":             sellers,
    "category_translation": category_translation,
}

print(f"{'Dataset':25s}  {'Rows':>10}  {'Columns':>8}")
print("-" * 48)
for name, df in datasets.items():
    print(f"{name:25s}  {df.shape[0]:>10,}  {df.shape[1]:>8}")

Dataset                          Rows   Columns
------------------------------------------------
customers                      99,441         5
geolocation                   738,307         5
order_items                   112,650         7
payments                      103,886         5
reviews                        99,224         7
orders                         99,441         8
products                       32,951         9
sellers                         3,095         4
category_translation               71         2


### 15b. Missing-value counts in cleaned datasets

In [105]:
print(f"{'Dataset':25s}  {'Missing cells':>15}")
print("-" * 43)
for name, df in datasets.items():
    total_missing = df.isna().sum().sum()
    print(f"{name:25s}  {total_missing:>15,}")

Dataset                      Missing cells
-------------------------------------------
customers                                0
geolocation                              0
order_items                              0
payments                                 0
reviews                            145,903
orders                               4,908
products                             2,452
sellers                                  0
category_translation                     0


### 15c. Remaining duplicate rows after cleaning

In [106]:
print(f"{'Dataset':25s}  {'Duplicate rows':>15}")
print("-" * 43)
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f"{name:25s}  {dupes:>15,}")

Dataset                     Duplicate rows
-------------------------------------------
customers                                0
geolocation                              0
order_items                              0
payments                                 0
reviews                                  0
orders                                   0
products                                 0
sellers                                  0
category_translation                     0


### 15d. Primary key uniqueness check

In [107]:
# Each of these columns should have zero duplicates (they are primary keys).
pk_checks = [
    ("customers",           customers,            "customer_id"),
    ("orders",              orders,               "order_id"),
    ("products",            products,             "product_id"),
    ("sellers",             sellers,              "seller_id"),
    ("category_translation", category_translation, "product_category_name"),
]

print(f"{'Dataset':25s}  {'Primary key column':30s}  {'Duplicates':>12}")
print("-" * 72)
for name, df, pk_col in pk_checks:
    dupes = df[pk_col].duplicated().sum()
    status = "OK" if dupes == 0 else "ISSUE"
    print(f"{name:25s}  {pk_col:30s}  {dupes:>10,}  {status}")

Dataset                    Primary key column                Duplicates
------------------------------------------------------------------------
customers                  customer_id                              0  OK
orders                     order_id                                 0  OK
products                   product_id                               0  OK
sellers                    seller_id                                0  OK
category_translation       product_category_name                    0  OK


### 15e. Final relationship check on cleaned data

In [108]:
clean_customers  = set(customers['customer_id'])
clean_orders     = set(orders['order_id'])
clean_products   = set(products['product_id'])
clean_sellers    = set(sellers['seller_id'])

checks = [
    ("orders → customers",           orders['customer_id'],      clean_customers),
    ("order_items → orders",         order_items['order_id'],    clean_orders),
    ("order_items → products",       order_items['product_id'],  clean_products),
    ("order_items → sellers",        order_items['seller_id'],   clean_sellers),
    ("payments → orders",            payments['order_id'],       clean_orders),
    ("reviews → orders",             reviews['order_id'],        clean_orders),
]

print(f"{'Relationship':35s}  {'Unmatched rows':>15}")
print("-" * 53)
for label, fk_series, parent_set in checks:
    bad = (~fk_series.isin(parent_set)).sum()
    status = "OK" if bad == 0 else "ISSUE"
    print(f"{label:35s}  {bad:>15,}  {status}")

Relationship                          Unmatched rows
-----------------------------------------------------
orders → customers                                 0  OK
order_items → orders                               0  OK
order_items → products                             0  OK
order_items → sellers                              0  OK
payments → orders                                  0  OK
reviews → orders                                   0  OK


---
## 16. Summary of resolved and unresolved issues

### What was resolved

| Dataset | Action taken |
|---|---|
| **Geolocation** | 261,831 exact duplicate rows removed. 50–55 rows with coordinates outside Brazil's geographic bounds removed. |
| **Order items** | `shipping_limit_date` converted to datetime. |
| **Reviews** | `review_creation_date` and `review_answer_timestamp` converted to datetime. |
| **Orders** | All 5 timestamp columns converted to datetime. |
| **Products** | 4 rows with `product_weight_g = 0` set to NaN (not a valid shipping weight). |
| **All datasets** | Text columns stripped of leading/trailing whitespace. |

### What remains unresolved (kept with flags)

| Dataset | Issue | Why we kept it |
|---|---|---|
| **Payments** | 9 rows with `payment_value = 0.00` | Could be vouchers or adjustments; cannot infer correct value. |
| **Payments** | 2 credit-card rows with `payment_installments = 0` | Suspicious but no reliable correction available. |
| **Payments** | 3 rows with `payment_type = 'not_defined'` | The orders are valid; the type label is unknown. |
| **Orders** | 1 delivered order with no payment record | Single anomaly; order is otherwise complete. |
| **Orders** | 160/1783/2965 missing timestamps | All explained by order status (cancelled, not shipped, etc.). |
| **Products** | 610 products missing category, name length, description length, photo count | These are real products sold in real orders; we cannot fill the values. |
| **Products** | 2 products missing weight and dimensions | Cannot infer correct values. |
| **Products** | 2 categories not in the translation table | `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` — missing translations only. |
| **Reviews** | 789 `review_id` values shared across different orders | System-level reuse of IDs; each row belongs to a real order. |
| **Customers** | 2,997 `customer_unique_id` values appearing more than once | Expected — these are returning customers, not errors. |

---
## Notebook complete

The nine cleaned CSV files are saved in `Cleaned_Data/`.  
A cleaning summary is at `Cleaned_Data/cleaning_summary.csv`.

These datasets are ready for Stage 1 (SQL, Python EDA, statistical analysis, Power BI) and Stage 2 (feature engineering, customer segmentation, machine learning).